# Validation: Function Pipeline Check

Verify all components work correctly before running hypothesis tests. Each function is tested independently on BTC 4H data.

In [ ]:
import sys
sys.path.append('../src')
from aggregation_data import load_and_resample

btc_4h = load_and_resample('../data/glbx-mdp3-20230101-20260131.ohlcv-1m.csv', '4h')
print(f"Data loaded: {btc_4h.shape[0]} bars, {btc_4h.shape[1]} columns")
print(f"Date range: {btc_4h.index[0]} to {btc_4h.index[-1]}")
print(btc_4h.head())

## Indicator Validation

In [ ]:
from indicators import hma, ssl_channels, alpha_trend, EMA200

# HMA
h = hma(btc_4h['high'], 60)
print(f"HMA(60): {h.dropna().shape[0]} valid values, first={h.dropna().iloc[0]:.2f}, last={h.dropna().iloc[-1]:.2f}")

# SSL Channels
hlv, hma_h, hma_l = ssl_channels(btc_4h, 60)
print(f"SSL(60): {int(sum(hlv == 1))} bullish bars, {int(sum(hlv == -1))} bearish bars, {int(sum(hlv == 0))} undecided")

# AlphaTrend
at = alpha_trend(btc_4h)
print(f"AlphaTrend: {sum(at > 0)} active bars, last value={at[-1]:.2f}")

# EMA 200
ema_c, cloud_up, cloud_low = EMA200(btc_4h)
print(f"EMA200 cloud: upper={cloud_up.iloc[-1]:.2f}, lower={cloud_low.iloc[-1]:.2f}, spread={cloud_up.iloc[-1] - cloud_low.iloc[-1]:.2f}")

## Signal Detection

In [ ]:
from signals import crossoverdetection

cross = crossoverdetection(btc_4h)
print(f"Crossovers detected: {int(sum(cross == 1))} bullish, {int(sum(cross == -1))} bearish, {int(sum(cross == 1) + sum(cross == -1))} total")

## Synthetic Data Generation

In [ ]:
from synthetic import generate_garch_series

sims = generate_garch_series(btc_4h['close'], 1)
print(f"Synthetic series length: {len(sims[0])}")
print(f"Real close range: {btc_4h['close'].min():.2f} - {btc_4h['close'].max():.2f}")
print(f"Synthetic range: {sims[0].min():.2f} - {sims[0].max():.2f}")

## Channel Test Event Detection

In [ ]:
from tests import compute_touch_returns

rets = compute_touch_returns(btc_4h, 60, 10)
print(f"Fresh channel tests: {len(rets)} events out of {len(btc_4h)} bars ({len(rets)/len(btc_4h)*100:.1f}%)")
print(f"Mean return after test: {sum(rets)/len(rets):.6f}")

## Validation Summary

All pipeline components functional. Ready for hypothesis testing.